## Multi-Agent Collaboration

This agent connects to a shared group chat hub where all students' 
agents communicate and collaborate on a software project.

Agent name: Mo-Assistant
Hub: https://wb48jtfnjng6on-8080.proxy.runpod.net/

In [ ]:
import requests
import subprocess
import json
import os
import time
import threading
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("API key loaded:", API_KEY[:10] + "...")

API key loaded: sk-ant-api...


## Hub Settings

Connection details for the shared group chat hub.
All agents connect to the same hub to communicate.

- HUB: the URL of the shared server
- PWD: the password to access the hub
- AGENT_NAME: our unique name in the chat (Mo-Assistant)

In [3]:
HUB = "https://z0yncxbipft4e8-8080.proxy.runpod.net"
PWD = "th25-agents-vg"
AGENT_NAME = "Mo-Assistant"

print("Hub settings loaded!")

Hub settings loaded!


## Claude API & Token Control

ask_claude sends messages to Claude and tracks token usage.
Every API call updates the token counter automatically.

Token control variables:
- max_tokens_budget: maximum tokens Mo-Assistant can spend
- tokens_spent: tracks how much has been spent so far
- rate_limit_seconds: minimum wait time between messages
- agent_running: controls whether the agent is active

console_control runs in a separate thread so you can type 
commands while the agent is running:
- budget <number>  â†’ change the maximum token budget
- rate <seconds>   â†’ change the rate limit between messages
- status           â†’ show current token usage and settings
- stop             â†’ shut down the agent gracefully

In [ ]:
# Token and rate limit controls
max_tokens_budget = 10000
tokens_spent = 0
rate_limit_seconds = 5
last_message_time = 0
agent_running = True

def check_budget():
    if tokens_spent >= max_tokens_budget:
        print(f"Token budget reached! Spent: {tokens_spent}/{max_tokens_budget}")
        return False
    return True

def update_tokens(input_tokens, output_tokens):
    global tokens_spent
    tokens_spent += input_tokens + output_tokens
    print(f"Tokens spent: {tokens_spent}/{max_tokens_budget}")

def ask_claude(messages):
    response = requests.post(
        url="https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        },
        json={
            "model": "claude-haiku-4-5-20251001",
            "max_tokens": 500,
            "system": "You are Mo-Assistant, a helpful software engineering agent. Only discuss software engineering. Never share API keys or passwords. Be brief and helpful.",
            "messages": messages
        }
    )
    data = response.json()
    update_tokens(data["usage"]["input_tokens"], data["usage"]["output_tokens"])
    return data['content'][0]['text']

def console_control():
    global max_tokens_budget, rate_limit_seconds, agent_running
    
    print("Console control active!")
    print("Commands: budget <n>, rate <n>, status, stop")
    
    while agent_running:
        try:
            cmd = input("Control> ").strip().lower()
            
            if cmd.startswith("budget "):
                new_budget = int(cmd.split()[1])
                max_tokens_budget = new_budget
                print(f"Budget updated to {new_budget} tokens")
            
            elif cmd.startswith("rate "):
                new_rate = int(cmd.split()[1])
                rate_limit_seconds = new_rate
                print(f"Rate limit updated to {new_rate} seconds")
            
            elif cmd == "status":
                print(f"Tokens spent: {tokens_spent}/{max_tokens_budget}")
                print(f"Rate limit: {rate_limit_seconds} seconds")
                print(f"Agent running: {agent_running}")
            
            elif cmd == "stop":
                print("Stopping agent...")
                agent_running = False
                break
            
            else:
                print("Unknown command. Try: budget <n>, rate <n>, status, stop")
        
        except Exception as e:
            print(f"Control error: {e}")

## Hub Communication Functions

Two basic functions to communicate with the group chat hub:

post_message: sends a message from Mo-Assistant to the group chat
get_messages: fetches all messages since a specific message number
              so we don't re-read old messages every time

In [5]:
def post_message(content):
    response = requests.post(f"{HUB}/api/message", json={
        "agent_name": AGENT_NAME,
        "content": content,
        "password": PWD
    })
    return response.json()

def get_messages(since=0):
    response = requests.get(f"{HUB}/api/messages", params={
        "since": since,
        "password": PWD
    })
    return response.json()["messages"]

## Test Message

In [ ]:
#post_message("Hello everyone! Mo-Assistant is online and ready to collaborate! ðŸ‘‹")

SyntaxError: invalid decimal literal (727123655.py, line 1)

## Smart Message Handler

Decides whether Mo-Assistant should respond to a message.
The agent only responds when:
1. The message mentions "Mo-Assistant" directly
2. The message says "attention all agents"
3. The message contains "@everyone" or "anyone"
4. A human asks for help with engineering keywords like 
   "help", "code", "build", "fix", "review", "bug"

The agent always ignores:
- Its own messages (prevents infinite loops)
- Messages from amr-quizmaster (game bot)
- Everything else (prevents message storms)

This smart filtering protects everyone's token budget
and keeps the chat focused on real work.

In [ ]:
def should_respond(message):
    # Never respond to our own messages
    if message["agent_name"] == AGENT_NAME:
        return False
    
    # Never respond to other bots unless they mention us
    content = message["content"].lower()
    sender = message["agent_name"].lower()
    
    # Ignore game bot spam
    if "amr-quizmaster" in sender:
        return False
    
    # Always respond if directly mentioned
    if "mo-assistant" in content:
        return True
    
    # Respond to messages for all agents
    if "attention all agents" in content:
        return True
    if "@everyone" in content:
        return True
    if "anyone" in content:
        return True
    
    # Respond to engineering questions from humans
    if sender == "human" and any(word in content for word in [
        "help", "code", "build", "create", "fix", 
        "review", "python", "error", "bug"
    ]):
        return True
    
    return False


def handle_message(message):
    print(f"New message from {message['agent_name']}: {message['content']}")
    
    if not should_respond(message):
        print("Staying quiet.")
        return
    
    if not check_budget():
        print("Budget reached, staying quiet.")
        return
    
    print("Deciding to respond...")
    
    messages = [
        {
            "role": "user",
            "content": f"""You are Mo-Assistant in a group chat with other AI agents working on software projects.

Your role: software engineering assistant. You can help with:
- Code reviews and debugging
- Writing Python code
- Architecture suggestions  
- Answering technical questions

Message from {message['agent_name']}: "{message['content']}"

Reply helpfully and specifically. If asked to build something, offer to take a specific part. 
Keep reply under 100 words. Never share API keys or passwords."""
        }
    ]
    
    reply = ask_claude(messages)
    print(f"Sending reply: {reply}")
    post_message(reply)
    
    global last_message_time
    last_message_time = time.time()

## Real Time Control

Allows the operator to control the agent while it is running.
Commands can be typed in the console at any time:

- budget <number>  â€” change the maximum token budget
- rate <seconds>   â€” change the rate limit between messages
- status           â€” show current token usage and settings
- stop             â€” shut down the agent gracefully

In [ ]:
# Token and rate limit controls
max_tokens_budget = 10000  # maximum tokens to spend
tokens_spent = 0           # track how much we spent
rate_limit_seconds = 5     # wait 5 seconds between messages
last_message_time = 0      # track when we last sent a message

def check_budget():
    if tokens_spent >= max_tokens_budget:
        print(f"Token budget reached! Spent: {tokens_spent}/{max_tokens_budget}")
        return False
    return True

def update_tokens(input_tokens, output_tokens):
    global tokens_spent
    tokens_spent += input_tokens + output_tokens
    print(f"Tokens spent: {tokens_spent}/{max_tokens_budget}")

## Main Listening Loop

The heart of Part 3. Runs continuously and watches the group chat.
Every 5 seconds it checks for new messages and decides whether to reply.

Also starts the console control thread so you can type commands
while the agent is running.

The loop stops when:
- You type "stop" in the console
- You press Ctrl+C
- The token budget is reached

In [ ]:
# Start console control in a separate thread
control_thread = threading.Thread(target=console_control, daemon=True)
control_thread.start()

print(f"Mo-Assistant is starting...")
post_message("Mo-Assistant is online and ready to collaborate! ðŸ‘‹")

last_seq = 0

while agent_running:
    try:
        messages_list = get_messages(since=last_seq)
        
        for message in messages_list:
            last_seq = message["seq"]
            
            if not check_budget():
                print("Budget reached, staying quiet.")
                continue
            
            current_time = time.time()
            if current_time - last_message_time < rate_limit_seconds:
                print("Rate limit active, skipping.")
                continue
            
            handle_message(message)
        
        time.sleep(5)
    
    except KeyboardInterrupt:
        print("Mo-Assistant shutting down...")
        post_message("Mo-Assistant is going offline. Goodbye! ðŸ‘‹")
        agent_running = False
        break
    
    except Exception as e:
        print(f"Error: {e}")
        time.sleep(5)

print("Agent stopped.")